# Lipid-droplet quantification

Raw Imaris IMS → native-intensity maximum projection → manual central-brain
area ROI in Fiji → guided 3D ROI → per-plane detection and cross-Z linking →
per-brain quantification and QC. Detections are not counted on the projection.

Use Python 3.11 and the packages in `requirements.txt`. Inputs and example
images are supplied separately; `example_data/` is currently a placeholder.
Results go to new subdirectories of `results/`. The statistical unit is a brain.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = None  # Set explicitly if the kernel starts outside this repository.
if PROJECT_ROOT is None:
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([parent, parent / 'Lipid droplet analysis'])
    PROJECT_ROOT = next((p for p in candidates if (p / 'script' / 'lipid_workflow.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Set PROJECT_ROOT to this analysis folder, then rerun this cell.')
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
script_dir = PROJECT_ROOT / 'script'
loaded = sys.modules.get('lipid_quant')
expected = script_dir / 'lipid_quant'
if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(script_dir):
    raise RuntimeError('Restart the kernel: a different copy of lipid_quant is already imported.')
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))
print('Analysis folder:', PROJECT_ROOT)


In [ ]:
from lipid_workflow import (
    load_workflow, prepare_guides, quantify, completed_result_dirs,
    supplement_qc, summarize,
)
from lipid_quant.batch_summary import display_batch_overview, display_report_outputs

CONFIG_FILE = PROJECT_ROOT / 'config' / 'workflow.json'
DATA_ROOT = PROJECT_ROOT / 'example_data'  # Change to the parent containing your input batches.
workflow = load_workflow(CONFIG_FILE, data_root=DATA_ROOT)
print('Inputs:', *workflow['inputs'], sep='\n')
print('Results:', workflow['results_root'])


## 1. Prepare manual central-brain annotations

`workflow.json` selects the input batches for every stage; `analysis.yaml`
contains the original ROI and detection parameters. First inspect the plan.
Set `RUN_EXPORT=True` to export raw, unscaled, level-0 MIPs. This writes
`central_brain_roi/` alongside each IMS, preserving existing projections and ROIs.

Open each TIFF in Fiji, draw one closed area ROI, and save it with the same
basename and `.roi` extension. Do not resize, crop, rotate or resample the
projection before drawing. The corresponding `guide_manifest.json` maps
each IMS to its own ROI. Manual annotation is required before quantification.

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    display(prepare_guides(workflow, run=True))
else:
    display(prepare_guides(workflow, run=False))


## 2. Quantify the selected batch

Inspect the plan for missing guides before enabling `RUN_QUANTIFICATION`.
Missing annotations are recorded as skipped; they are not replaced by an
automatic ROI. Set `export_final_cb_mask` in the workflow config if the
final analysis ROI should also be exported as a full-resolution stack.

In [ ]:
RUN_QUANTIFICATION = False
batch_status = quantify(workflow, run=RUN_QUANTIFICATION)
display(batch_status)
selected_results = (
    completed_result_dirs(batch_status) if RUN_QUANTIFICATION
    else workflow['existing_result_dirs']
)
print('Selected completed results:', len(selected_results))


## 3. Supplement QC from saved detections

This uses the completed outputs selected above. To inspect an existing
run after restarting the kernel, set `existing_result_dirs` in `workflow.json`
and reload the configuration. Saved object IDs and coordinates supply the
QC markers; detection is not repeated. Raw IMS is needed for image overlays.

In [ ]:
RUN_QC = False
if selected_results:
    display(supplement_qc(workflow, selected_results, run=RUN_QC))
else:
    print('Run quantification or configure existing_result_dirs before QC.')


## 4. Summarize and export

Group labels, order, selected comparisons and statistical options are in
`workflow.json`. Duplicate brain IDs are rejected rather than counted as
independent samples. Review ROI scope, review flags and configuration
differences before pooling batches. LD-positive core volume and equivalent-
sphere volume retain their original, different definitions.

In [ ]:
RUN_SUMMARY = False
report = None
if RUN_SUMMARY:
    report = summarize(workflow, selected_results, show=True)
    display_batch_overview(report['results'], summary=report['summary'], validation=report['validation'])
    display_report_outputs(report['report'])
    print('Report:', report['output_dir'])
